# Notebook 03 — Field Boundary & NRCS Soil EDA

This notebook explores the Iowa field boundary GeoJSON and the paired NRCS
SSURGO soil survey data generated by `scripts/download_soil.py`.

**Data sources**
- `data/fields_EPSG4326.geojson` — 20 Iowa field boundary polygons (USDA NASS CDL)
- `data/soil_EPSG4326.csv` — NRCS SSURGO soil attributes per field
- `data/fields_with_soil.geojson` — field boundaries enriched with soil data


In [ ]:
import json
import csv
from pathlib import Path
import pprint
from collections import Counter, defaultdict

# ── Paths (relative to project root; adjust if running from notebooks/) ──────
ROOT             = Path("..")
FIELDS_GEOJSON   = ROOT / "data" / "fields_EPSG4326.geojson"
SOIL_CSV         = ROOT / "data" / "soil_EPSG4326.csv"
ENRICHED_GEOJSON = ROOT / "data" / "fields_with_soil.geojson"

print("Files found:", {p.name: p.exists() for p in [FIELDS_GEOJSON, SOIL_CSV, ENRICHED_GEOJSON]})

## 1  Load field boundaries

In [ ]:
with open(FIELDS_GEOJSON) as f:
    fields_geojson = json.load(f)

features = fields_geojson["features"]
print(f"Total field features: {len(features)}")
pprint.pprint(features[0])

In [ ]:
crop_counts   = Counter(f["properties"]["crop"]   for f in features)
region_counts = Counter(f["properties"]["region"] for f in features)
areas         = [f["properties"]["area_acres"] for f in features]

print("Crop distribution:  ", dict(crop_counts))
print("Region distribution:", dict(region_counts))
print(f"Field area — min: {min(areas)} ac, max: {max(areas)} ac, mean: {sum(areas)/len(areas):.1f} ac")

## 2  Load NRCS SSURGO soil data

In [ ]:
soil_records = []
with open(SOIL_CSV, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        soil_records.append({
            "field_id":       int(row["field_id"]),
            "soil_type":      row["soil_type"],
            "mapunit_name":   row.get("mapunit_name", ""),
            "drainage_class": row.get("drainage_class", ""),
            "slope_pct":      float(row["slope_pct"]) if row.get("slope_pct") else None,
            "ph":             float(row["ph"]),
            "organic_matter": float(row["organic_matter"]),
        })

print(f"Soil records loaded: {len(soil_records)}")
print("\nSample records (first 5):")
for r in soil_records[:5]:
    print(" ", r)

In [ ]:
soil_type_counts   = Counter(r["soil_type"]   for r in soil_records)
drainage_counts    = Counter(r["drainage_class"] for r in soil_records)
ph_values          = [r["ph"]             for r in soil_records]
om_values          = [r["organic_matter"] for r in soil_records]

print("Soil type distribution:")
for st, n in sorted(soil_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {st}: {n}")

print("\nDrainage class distribution:")
for dc, n in sorted(drainage_counts.items(), key=lambda x: -x[1]):
    print(f"  {dc}: {n}")

print(f"\npH            — min: {min(ph_values):.2f}, max: {max(ph_values):.2f}, mean: {sum(ph_values)/len(ph_values):.2f}")
print(f"Organic matter — min: {min(om_values):.2f}, max: {max(om_values):.2f}, mean: {sum(om_values)/len(om_values):.2f}%")

## 3  Inspect enriched GeoJSON

In [ ]:
with open(ENRICHED_GEOJSON) as f:
    enriched = json.load(f)

enriched_features = enriched["features"]
print(f"Enriched features: {len(enriched_features)}")

# Confirm soil property is present
soil_key = "soil" if "soil" in enriched_features[0]["properties"] else "nrcs_soil"
with_soil = [f for f in enriched_features if soil_key in f["properties"]]
print(f"Features with '{soil_key}': {len(with_soil)}")
pprint.pprint(enriched_features[0]["properties"])

## 4  Cross-analysis: crop vs. soil type

In [ ]:
crop_soil: dict[str, list[str]] = defaultdict(list)
crop_ph:   dict[str, list[float]] = defaultdict(list)
crop_om:   dict[str, list[float]] = defaultdict(list)

for feat in enriched_features:
    props = feat["properties"]
    soil = props.get(soil_key, {})
    if soil:
        crop = props["crop"]
        crop_soil[crop].append(soil["soil_type"])
        crop_ph[crop].append(soil["ph"])
        crop_om[crop].append(soil["organic_matter"])

print("Average soil attributes by crop:")
for crop in sorted(crop_ph):
    avg_ph = sum(crop_ph[crop]) / len(crop_ph[crop])
    avg_om = sum(crop_om[crop]) / len(crop_om[crop])
    soil_dist = Counter(crop_soil[crop])
    top_soil  = max(soil_dist, key=soil_dist.get)
    print(
        f"  {crop:10s}  pH={avg_ph:.2f}  organic_matter={avg_om:.2f}%  "
        f"top_soil={top_soil} ({soil_dist[top_soil]}/{len(crop_ph[crop])})"
    )